<a href="https://colab.research.google.com/github/FabianJerez/TP1/blob/master/5_CIA_EDA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 5 · Guía de trabajo en equipo — Correlación (Pearson vs. Spearman) y heatmap


> **¿Dónde estamos en el flujo del EDA?**
> En el análisis bivariado **vimos** relaciones con gráficos de dispersión. Ahora vamos a **medirlas con un número**: el **coeficiente de correlación**. Seguimos en el paso **05 – Explorar y analizar** (correlaciones).
> Teoría relacionada: láminas *"Tipos de correlación"*, *"La correlación no implica causalidad"* y *"Valores atípicos"*.

> **Pregunta guía del análisis (la misma de los notebooks 0 y 1):**
> ¿Qué características de las células parecen diferenciar un tumor **maligno** (`target = 0`) de uno **benigno** (`target = 1`)?

> **Cómo trabajar**
> 1. Hagan una copia del notebook (*Archivo → Guardar una copia en Drive*) y suban `sample_dataset.csv` al panel **Archivos** (ícono de carpeta, a la izquierda).
> 2. Ejecuten las celdas **en orden** (Shift + Enter).
> 3. Lean los cuadros 📘, completen las celdas 🧩 (reemplazar cada `COMPLETAR`) y respondan en las celdas ✍️.
>    Las celdas 🧩 **dan error hasta que reemplacen `COMPLETAR`**: es esperable, no es un problema del notebook.
> 4. Al final, armen la sección 🎤: **cada equipo va a explicar su notebook al resto del curso en 5 minutos.**
>
> | Ícono | Significa |
> |---|---|
> | 📘 | Concepto (relacionado con la teoría vista) |
> | 💻 | Código listo para ejecutar y observar |
> | 🧩 | Código para completar (reemplazar `COMPLETAR`) |
> | ✍️ | Pregunta para responder en equipo |
> | ⭐ | Extra, solo si les sobra tiempo |
> | 🎤 | Lo que van a presentar al resto |

## 0) Preparación

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# Cada notebook es independiente: hay que volver a importar librerías y cargar los datos.
# Si aparece FileNotFoundError: el CSV no está subido en el panel "Archivos" de ESTE notebook.
df = pd.read_csv("sample_dataset.csv")
df.shape   # esperamos (569, 31)

## 1) El coeficiente de correlación

📘 **Concepto.** El coeficiente de correlación **r** resume en un número **la dirección y la fuerza** de la relación entre dos variables numéricas:

| Valor de r | Interpretación |
|---|---|
| r = 1 | positiva **perfecta** |
| r ≈ 0,7 a 0,99 | positiva **fuerte** |
| r ≈ 0,3 a 0,7 | positiva **moderada** |
| r ≈ 0 | **nula** (sin relación *lineal*) |
| r negativos | lo mismo, pero la relación es **negativa** (una sube, la otra baja) |

*Los cortes son orientativos: dependen del área de estudio.*

In [ ]:
# Correlación entre dos columnas (los NaN se ignoran de a pares)
r1 = df["mean perimeter"].corr(df["mean area"])
print(f"mean perimeter vs mean area: r = {r1:.2f}")

🧩 **Completen** para calcular la correlación entre `mean radius` y `mean smoothness`.

In [ ]:
r2 = df["mean radius"].corr(df["mean smoothness"])
print(f"mean radius vs mean smoothness: r = {r2:.2f}")

✍️ **Pregunta:** Usando la tabla, ¿cómo clasifican cada una de las dos correlaciones?

✍️ **Respuesta del equipo:**

_(Doble clic acá para escribir la respuesta.)_

## 2) La matriz de correlación

💻 `df.corr()` calcula la correlación **de todas las variables contra todas**.

> **Ojo (pandas 2.x, la versión de Colab):** si el DataFrame tiene columnas de texto (como `area error`), `df.corr()` da **error**. Por eso agregamos `numeric_only=True`, que usa solo las columnas numéricas.

In [ ]:
corr = df.corr(numeric_only=True)
corr.round(2)

✍️ **Pregunta:** ¿Por qué la diagonal está llena de 1? ¿Por qué la matriz es simétrica? ¿Es fácil sacar conclusiones mirando esta tabla?

✍️ **Respuesta del equipo:**

_(Doble clic acá para escribir la respuesta.)_

## 3) Pearson vs. Spearman

📘 **Concepto.** Por defecto, `corr()` calcula el coeficiente de **Pearson**. Hay otro muy usado: **Spearman**.

| Característica | **Pearson** | **Spearman** |
|---|---|---|
| Mide relaciones... | **lineales** (línea recta) | **monótonas** (siempre crece o siempre decrece, aunque sea en curva) |
| Se calcula sobre... | los valores originales | los **rangos** (posiciones al ordenar los datos: 1°, 2°, 3°...) |
| Sensibilidad a outliers | **alta** | **baja** |
| Supone distribución aprox. normal | sí | no |
| Uso típico | altura vs. peso | ranking de satisfacción vs. edad; datos asimétricos u ordinales |

**Idea clave:** como Spearman trabaja con posiciones, un valor extremo es "solo el primero de la fila", no un número gigante que arrastra el cálculo.

In [ ]:
corr_p = df.corr(method="pearson", numeric_only=True)
corr_s = df.corr(method="spearman", numeric_only=True)

for a, b in [("mean radius", "mean area"), ("mean concavity", "mean area"), ("radius error", "mean radius")]:
    print(f"{a:>15} vs {b:<12}  Pearson = {corr_p.loc[a, b]:.2f} | Spearman = {corr_s.loc[a, b]:.2f}")

## 4) Heatmap: la matriz de correlación en colores

📘 **Concepto.** Un **heatmap** (mapa de calor) pinta cada celda de una matriz según su valor. Con la paleta `coolwarm` y la escala fija entre -1 y 1:
- **Rojo intenso** → correlación positiva fuerte
- **Azul intenso** → correlación negativa fuerte
- **Blanco / gris claro** → cerca de 0, sin relación lineal

`annot=True` escribe el valor dentro de cada celda (útil con pocas variables).

In [ ]:
cols_mean = [c for c in df.columns if c.startswith("mean")]   # solo las 10 variables "mean ..."

plt.figure(figsize=(10, 8))
sns.heatmap(df[cols_mean].corr(), annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1)
plt.title("Correlación (Pearson) entre las variables 'mean'")
plt.show()

✍️ **Preguntas**
1. ¿Qué pares tienen correlación **mayor a 0,9**? ¿Forman "grupos" o "familias" de variables?
2. ¿Hay alguna correlación **negativa** que se destaque?

✍️ **Respuesta del equipo:**

_(Doble clic acá para escribir la respuesta.)_

In [ ]:
# Todas las variables numéricas (sin números: con 30 variables no se leerían)
plt.figure(figsize=(14, 11))
sns.heatmap(corr, cmap="coolwarm", vmin=-1, vmax=1)
plt.title("Matriz de correlación completa (Pearson)")
plt.show()

💻 **Heatmap en valor absoluto.** Si nos interesa solo la **intensidad** de la correlación (sin importar si es positiva o negativa), graficamos el valor absoluto: así una correlación de -0,8 y una de 0,8 se ven igual de "fuertes".

In [ ]:
#Si nos interesa solo la intensidad de la correlación, podemos graficar con el valor absoluto
plt.figure(figsize=(12,8))
sns.heatmap(corr.abs(), cmap="Reds", vmin=0, vmax=1)
plt.show()

✍️ **Pregunta:** ¿Qué ganamos y qué perdemos al usar el valor absoluto?

✍️ **Respuesta del equipo:**

_(Doble clic acá para escribir la respuesta.)_

## 5) Volvamos a la pregunta guía: ¿qué variables se relacionan más con el diagnóstico?

📘 **Ojo con el signo.** `target` vale **0 = maligno** y **1 = benigno**. Entonces:
- correlación **negativa** con `target` → cuando la variable **sube**, es más probable que sea **maligno**;
- correlación **positiva** → valores altos se asocian a **benigno**.

*(`target` es binaria, así que esta correlación se usa como una medida **orientativa** de asociación.)*

In [ ]:
corr_target = corr["target"].drop("target").sort_values()

corr_target.plot(kind="barh", figsize=(7, 10), color=np.where(corr_target < 0, "tab:red", "tab:blue"))
plt.axvline(0, color="black", linewidth=0.8)
plt.title("Correlación de cada variable con target\n(negativo = asociada a maligno)")
plt.show()

corr_target.head(10).round(2)

## 6) Para pensar: dos advertencias

📘 1) **Correlación no implica causalidad.** 2) **Los outliers pueden cambiar mucho el valor de r** (lámina: un mismo conjunto puede dar r = 0,2 con outliers y r = 0,8 sin ellos).

✍️ **Pregunta:** `worst concave points` tiene una correlación fuerte con el diagnóstico. ¿Podemos decir que los puntos cóncavos **causan** el cáncer? ¿Cómo lo dirían correctamente en un informe?

✍️ **Respuesta del equipo:**

_(Doble clic acá para escribir la respuesta.)_